# 03 · Covariates, polygenic scores and cessation outcome

Paper sections: **Methods → Polygenic Score, Covariates, Social Determinants of Health, Self-reported Smoking Behaviors**.

Builds one person-level covariate table (demographics, ancestry-specific PGS + 10 PCs, deprivation index,
survey SDoH, psychiatric comorbidities), joins it to the treated cohort, and adds the self-reported cessation outcome.

**Inputs (workspace bucket `data/`):**
- `cohort_treatment.csv` (notebook 02)
- PLINK score files: `prs_cig_per_day.csv`, `prs_cig_afrV2.csv`, `prs_ces_eur.csv`, `prs_ces_afr.csv` (PRS-CSx, see README)
- PCs: `eur_pca.eigenvec`, `ancestry_specific_pcs/afr_pca.eigenvec`
- `psychiatric_diagnoses_EHR_V2.csv` (see README)

**Outputs:** `person_covariates.csv`, `DF_smokerV2.csv` (full cohort + covariates), `DF_smokerV3.csv` (analytic sample with outcome)

In [ ]:
# Setup: run from the repository root inside the All of Us Researcher Workbench
source("R/helpers.R")
install_if_missing(c("tidyverse", "bigrquery", "allofus"))
con <- aou_connect()

## 1. Demographics

In [ ]:
person_tbl <- tbl(con,"person")

demographics <- person_tbl %>% 
  select(person_id, birth_datetime, sex_at_birth_source_concept_id, race_concept_id, ethnicity_concept_id) %>%
  collect() %>%
  mutate(
    person_id = as.double(person_id),
    # Calculate age at index date
    birth_date = as.Date(birth_datetime),
    sex = case_when(
      sex_at_birth_source_concept_id == 1585846 ~ "Male",
      sex_at_birth_source_concept_id == 1585847 ~ "Female",
      TRUE ~ "Other"
    ),
    race = case_when(
      race_concept_id == 8515 ~ "Asian",
      race_concept_id == 8516 ~ "Black or African American",
      race_concept_id == 8527 ~ "White",
      race_concept_id == 8557 ~ "Native Hawaiian or Other Pacific Islander",
      race_concept_id == 8657 ~ "American Indian or Alaska Native",
      race_concept_id == 38003615 ~ "Middle Eastern or North African",
      race_concept_id == 2000000008 ~ "More than one population",
      TRUE ~ "Other"
    ),
    ethnicity = case_when(
      ethnicity_concept_id == 38003564 ~ "Not Hispanic or Latino",
      ethnicity_concept_id == 38003563 ~ "Hispanic or Latino",
      TRUE ~ "Other"
    )
  )

## 2. Polygenic scores and ancestry PCs

### PGS for cigarettes per day (CPD<sup>PGS</sup>)

In [ ]:
prs_cig_eur <- load_from_bucket("prs_cig_per_day.csv")
colnames(prs_cig_eur) <- "data"
prs_cig_eur <- prs_cig_eur %>%
  separate(data, into = c("FID", "IID", "PHENO", "CNT", "CNT2", "SCORE"), sep = "\\s+") %>%
  rename(PERSON_ID = IID)

prs_cig_afr <- load_from_bucket("prs_cig_afrV2.csv")
colnames(prs_cig_afr) <- "data"
prs_cig_afr <- prs_cig_afr %>%
  separate(data, into = c("FID", "IID", "PHENO", "CNT", "CNT2", "SCORE"), sep = "\\s+") %>%
  rename(PERSON_ID = IID)

In [ ]:
####################################################
# EUR PCs and PRS
####################################################

eur_PCs <- load_from_bucket("eur_pca.eigenvec", reader = read_delim, 
                      delim="\t", 
                      skip = 1,
                      col_names=c("FID", "IID", paste0("PC", 1:10))) %>%
  rename(PERSON_ID = IID) %>%
  mutate(PERSON_ID = as.character(PERSON_ID),
         ancestry = "EUR") %>%
  select(PERSON_ID, ancestry, PC1:PC10)

# EUR PRS - PERSON_ID already exists, don't rename IID again!
prs_cig_eur <- prs_cig_eur %>%
  rename(PRS_CPD_EUR = SCORE) %>%  # Just rename SCORE
  mutate(PERSON_ID = as.character(PERSON_ID),
         PRS_CPD_EUR = as.numeric(PRS_CPD_EUR),
         PRS_CPD_EUR_scaled = scale(PRS_CPD_EUR)[,1]) %>%
  left_join(eur_PCs, by = "PERSON_ID") %>%
  select(PERSON_ID, ancestry, PRS_CPD_EUR, PRS_CPD_EUR_scaled, PC1:PC10)

####################################################
# AFR PCs and PRS
####################################################
# NOTE (cleanup): the original read the AFR PCs from a different workspace bucket.
# Copy afr_pca.eigenvec into this workspace's bucket at data/ancestry_specific_pcs/.
afr_PCs <- load_from_bucket("ancestry_specific_pcs/afr_pca.eigenvec", reader = read_delim, 
                      delim="\t", 
                      skip = 1,
                      col_names=c("FID", "IID", paste0("PC", 1:10))) %>%
  rename(PERSON_ID = IID) %>%
  mutate(PERSON_ID = as.character(PERSON_ID),
         ancestry = "AFR") %>%
  select(PERSON_ID, ancestry, PC1:PC10)

# AFR PRS - PERSON_ID already exists, don't rename IID again!
prs_cig_afr <- prs_cig_afr %>%
  rename(PRS_CPD_AFR = SCORE) %>%  # Just rename SCORE
  mutate(PERSON_ID = as.character(PERSON_ID),
         PRS_CPD_AFR = as.numeric(PRS_CPD_AFR),
         PRS_CPD_AFR_scaled = scale(PRS_CPD_AFR)[,1]) %>%
  left_join(afr_PCs, by = "PERSON_ID") %>%
  select(PERSON_ID, ancestry, PRS_CPD_AFR, PRS_CPD_AFR_scaled, PC1:PC10)

####################################################
# Combine EUR and AFR
####################################################
prs_combined <- bind_rows(
  prs_cig_eur %>% 
    rename(PRS_CPD = PRS_CPD_EUR,
           PRS_CPD_scaled = PRS_CPD_EUR_scaled),
  prs_cig_afr %>% 
    rename(PRS_CPD = PRS_CPD_AFR,
           PRS_CPD_scaled = PRS_CPD_AFR_scaled)
)

table(prs_combined$ancestry)

### PGS for smoking cessation (CES<sup>PGS</sup>)

In [ ]:
prs_ces_eur <- load_from_bucket("prs_ces_eur.csv")
colnames(prs_ces_eur) <- "data"
prs_ces_eur <- prs_ces_eur %>%
  separate(data, into = c("FID", "IID", "PHENO", "CNT", "CNT2", "SCORE"), sep = "\\s+") %>%
  rename(PERSON_ID = IID)

prs_ces_afr <- load_from_bucket("prs_ces_afr.csv")
colnames(prs_ces_afr) <- "data"
prs_ces_afr <- prs_ces_afr %>%
  separate(data, into = c("FID", "IID", "PHENO", "CNT", "CNT2", "SCORE"), sep = "\\s+") %>%
  rename(PERSON_ID = IID)

In [ ]:
####################################################
# Process CES PRS for EUR
####################################################
prs_ces_eur_processed <- prs_ces_eur %>%
  select(PERSON_ID, SCORE) %>%
  mutate(PERSON_ID = as.character(PERSON_ID),
         PRS_CES = as.numeric(SCORE),
         PRS_CES_scaled = scale(PRS_CES)[,1],
         ancestry = "EUR") %>%
  select(PERSON_ID, ancestry, PRS_CES, PRS_CES_scaled)

####################################################
# Process CES PRS for AFR
####################################################
prs_ces_afr_processed <- prs_ces_afr %>%
  select(PERSON_ID, SCORE) %>%
  mutate(PERSON_ID = as.character(PERSON_ID),
         PRS_CES = as.numeric(SCORE),
         PRS_CES_scaled = scale(PRS_CES)[,1],
         ancestry = "AFR") %>%
  select(PERSON_ID, ancestry, PRS_CES, PRS_CES_scaled)

# Combine CES PRS
prs_ces_combined <- bind_rows(prs_ces_eur_processed, prs_ces_afr_processed)

####################################################
# Add CES to your existing PRS_combined dataset
####################################################
prs_combined_full <- prs_combined %>%
  left_join(
    prs_ces_combined %>% select(PERSON_ID, PRS_CES, PRS_CES_scaled),
    by = "PERSON_ID"
  )

prs_combined_full <- prs_combined_full %>%
  mutate(person_id = as.double(PERSON_ID)) %>%
  select(-PERSON_ID)

## 3. Social determinants of health

### Geographic: deprivation index

In [ ]:
geo <- tbl(con, "ds_zip_code_socioeconomic") %>%
  select(PERSON_ID, DEPRIVATION_INDEX) %>%
  collect() %>%
  mutate(person_id = as.double(PERSON_ID)) %>%
  select(person_id, DEPRIVATION_INDEX) %>%
  distinct(person_id, .keep_all = TRUE)

### Individual: education, income, employment, insurance (The Basics survey)

In [ ]:
library(tidyverse)
library(bigrquery)
# This query represents dataset "edu_income_depindex" for domain "survey" and was generated for All of Us Controlled Tier Dataset v8
dataset_59120991_survey_sql <- paste("
    SELECT
        answer.person_id,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer  
    FROM
        `ds_survey` answer   
    WHERE
        (
            question_concept_id IN (SELECT
                DISTINCT concept_id                         
            FROM
                `cb_criteria` c                         
            JOIN
                (SELECT
                    CAST(cr.id as string) AS id                               
                FROM
                    `cb_criteria` cr                               
                WHERE
                    concept_id IN (1586134)                               
                    AND domain_id = 'SURVEY') a 
                    ON (c.path like CONCAT('%', a.id, '.%'))                         
            WHERE
                domain_id = 'SURVEY'                         
                AND type = 'PPI'                         
                AND subtype = 'QUESTION')
        )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
survey_59120991_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "survey_59120991",
  "survey_59120991_*.csv")
message(str_glue('The data will be written to {survey_59120991_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_59120991_survey_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  survey_59120991_path,
  destination_format = "CSV")
# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {survey_59120991_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(survey = col_character(), question = col_character(), answer = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_59120991_survey_df <- read_bq_export_from_workspace_bucket(survey_59120991_path)

In [ ]:
edu <- subset(dataset_59120991_survey_df, question == "Education Level: Highest Grade")

edu$edu[edu$answer == "Highest Grade: Never Attended"] <- 0
edu$edu[edu$answer == "Highest Grade: One Through Four"] <- 1
edu$edu[edu$answer == "Highest Grade: Five Through Eight"] <- 2
edu$edu[edu$answer == "Highest Grade: Nine Through Eleven"] <- 3
edu$edu[edu$answer == "Highest Grade: Twelve Or GED"] <- 4
edu$edu[edu$answer == "Highest Grade: College One to Three"] <- 5
edu$edu[edu$answer == "Highest Grade: College Graduate"] <- 6
edu$edu[edu$answer == "Highest Grade: Advanced Degree"] <- 7

In [ ]:
income <- subset(dataset_59120991_survey_df, question == "Income: Annual Income")

income$income[income$answer == "Annual Income: less 10k"] <- 0
income$income[income$answer == "Annual Income: 10k 25k"] <- 1
income$income[income$answer == "Annual Income: 25k 35k"] <- 2
income$income[income$answer == "Annual Income: 35k 50k"] <- 3
income$income[income$answer == "Annual Income: 50k 75k"] <- 4
income$income[income$answer == "Annual Income: 75k 100k"] <- 5
income$income[income$answer == "Annual Income: 100k 150k"] <- 6
income$income[income$answer == "Annual Income: 150k 200k"] <- 7
income$income[income$answer == "Annual Income: more 200k"] <- 8

In [ ]:
employment <- subset(dataset_59120991_survey_df, question == "Employment: Employment Status")

employment <- employment %>%
  distinct(person_id, .keep_all = TRUE)

employment$employment_status <- NA
employment$employment_status[employment$answer == "Employment Status: Unable To Work"] <- 0
employment$employment_status[employment$answer == "Employment Status: Out Of Work One Or More"] <- 0
employment$employment_status[employment$answer == "Employment Status: Out Of Work Less Than One"] <- 0
employment$employment_status[employment$answer == "Employment Status: Student"] <- 1
employment$employment_status[employment$answer == "Employment Status: Homemaker"] <- 2
employment$employment_status[employment$answer == "Employment Status: Self Employed"] <- 3
employment$employment_status[employment$answer == "Employment Status: Employed For Wages"] <- 4
employment$employment_status[employment$answer == "Employment Status: Retired"] <- 5

In [ ]:
insurance <- subset(dataset_59120991_survey_df, question == "Health Insurance: Insurance Type Update")

insurance <- insurance %>%
  distinct(person_id, .keep_all = TRUE)

insurance$insurance_type <- NA

insurance$insurance_type[insurance$answer == "Insurance Type Update: None"] <- 0
insurance$insurance_type[insurance$answer == "Insurance Type Update: Medicaid"] <- 1
insurance$insurance_type[insurance$answer == "Insurance Type Update: Purchased"] <- 2
insurance$insurance_type[insurance$answer == "Insurance Type Update: Other Health Plan"] <- 3
insurance$insurance_type[insurance$answer == "Insurance Type Update: Medicare"] <- 4
insurance$insurance_type[insurance$answer == "Insurance Type Update: Employer Or Union"] <- 5
insurance$insurance_type[insurance$answer == "Insurance Type Update: Military"] <- 6
insurance$insurance_type[insurance$answer == "Insurance Type Update: VA"] <- 6
insurance$insurance_type[insurance$answer == "Insurance Type Update: Indian"] <- 6

insurance$insurance_cat <- NA

# 0 = No insurance
insurance$insurance_cat[insurance$answer == "Insurance Type Update: None"] <- 0

# 1 = Public insurance (means-tested)
insurance$insurance_cat[insurance$answer == "Insurance Type Update: Medicaid"] <- 1

# 2 = Public insurance (Medicare/Government)
insurance$insurance_cat[insurance$answer == "Insurance Type Update: Medicare"] <- 2
insurance$insurance_cat[insurance$answer == "Insurance Type Update: Military"] <- 2
insurance$insurance_cat[insurance$answer == "Insurance Type Update: VA"] <- 2
insurance$insurance_cat[insurance$answer == "Insurance Type Update: Indian"] <- 2

# 3 = Private insurance
insurance$insurance_cat[insurance$answer == "Insurance Type Update: Employer Or Union"] <- 3
insurance$insurance_cat[insurance$answer == "Insurance Type Update: Purchased"] <- 3
insurance$insurance_cat[insurance$answer == "Insurance Type Update: Other Health Plan"] <- 3

## 4. Psychiatric comorbidities

EHR-derived indicators for schizophrenia, MDD, ADHD, bipolar disorder, anxiety and PTSD (see README for how this file is built).

In [ ]:
psychiatric_diagnoses_EHR_V2 <- load_from_bucket("psychiatric_diagnoses_EHR_V2.csv")

## 5. Person-level covariate table

One row per participant; reused by the varenicline analyses in notebook 07.

In [ ]:
# NOTE (cleanup): the original joined `insurance_type` here but fit models on
# `insurance_cat`; both are carried forward now.
person_covariates <- demographics %>%
  left_join(prs_combined_full, by = "person_id") %>%
  left_join(geo, by = "person_id") %>%
  left_join(employment %>% select(person_id, employment_status), by = "person_id") %>%
  left_join(income %>% select(person_id, income), by = "person_id") %>%
  left_join(insurance %>% select(person_id, insurance_type, insurance_cat), by = "person_id") %>%
  left_join(edu %>% select(person_id, edu), by = "person_id") %>%
  left_join(psychiatric_diagnoses_EHR_V2 %>%
              select(person_id, SCZ, ADHD, Bipolar_Disorder, Anxiety, Depression, PTSD),
            by = "person_id")

save_to_bucket(person_covariates, "person_covariates.csv")

## 6. Full cohort with covariates

In [ ]:
DF_smoker <- load_from_bucket("cohort_treatment.csv")

DF_smokerV2 <- DF_smoker %>%
  left_join(person_covariates, by = "person_id") %>%
  mutate(age = as.numeric(difftime(earliest_date, birth_datetime, units = "days")) / 365)

table(DF_smokerV2$ancestry, useNA = "ifany")
save_to_bucket(DF_smokerV2, "DF_smokerV2.csv")

## 7. Cessation outcome (Lifestyle survey)

“Do you now smoke cigarettes every day, some days, or not at all?”: *Not at all* = quit; *Every day / Some days* = current smoker. The first survey response after the cohort entry date is used.

In [ ]:
library(allofus)
con <- aou_connect()

cohort <- DF_smokerV2 %>%
  distinct(person_id) 

survey <- aou_survey(
  cohort,
  questions = c(1585867, 1585860),
  collect = T
)

In [ ]:
survey_outcomes <- survey %>%
  select(person_id, 
         Smoking_SmokeFrequency_date,
         Smoking_SmokeFrequency,
         Smoking_SeriousQuitAttempt,
         Smoking_SeriousQuitAttempt_date
         ) %>%
  mutate(
    smoking_status = case_when(
      Smoking_SmokeFrequency == "EveryDay" | Smoking_SmokeFrequency == "SomeDays" ~ "current_smoker",
      Smoking_SmokeFrequency == "NotAtAll" ~ "quit_success",
      TRUE ~ "unknown"
    ),
    quit_attempt = case_when(
      Smoking_SeriousQuitAttempt == "AttemptQuitSmoking" ~ "yes",
      Smoking_SeriousQuitAttempt == "NoAttemptQuitSmoking" ~ "no",
      TRUE ~ "unknown"
    )
  )

In [ ]:
survey_outcomes <- survey_outcomes %>%
mutate(person_id = as.double(person_id))

In [ ]:
DF_smokerV3 <- DF_smokerV2 %>%
  inner_join(survey_outcomes, by = "person_id") %>%
  mutate(
    days_from_index = as.numeric(difftime(Smoking_SmokeFrequency_date, earliest_date, units = "days"))
  )%>%
  filter(days_from_index > 0) %>%
  group_by(person_id) %>%
  slice(1) %>%  # Take first record per person
  ungroup()

In [ ]:
library(dplyr)
DF_smokerV3 <- DF_smokerV3 %>%
  filter(smoking_status != "unknown") %>%
  mutate(quit_success = ifelse(smoking_status == "quit_success", 1, 0))

table(DF_smokerV3$quit_success)
save_to_bucket(DF_smokerV3, "DF_smokerV3.csv")